<a href="https://colab.research.google.com/github/mayank261193/Session-6-Pandas/blob/main/S3_groupby_merge_and_challenge_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# S06 — Practice 3: GroupBy, Merge & Challenge
### SOLUTIONS

**Applied AI & Analytics Lab** · Session 6 · Practice notebook 3 of 3

**Time:** about 25 minutes.
**How to use this:** each exercise has a cell with `____` blanks or a `# YOUR CODE HERE` line. Fill it in and run it. The expected output is in the comment.

> **Setup:** upload `session6_policy_data.csv` and `session6_region_lookup.csv` (folder icon → Upload) before you start.

Do not open the solutions notebook until you have tried every one. Struggling **is** the learning.

---

> **Solutions notebook.** Use this only after you have really tried.
> Reading a solution feels like learning. It is not. Struggling is.

In [21]:
import pandas as pd
df = pd.read_csv("session6_policy_data.csv")
# clean first, so groups and merges are correct
df["region"] = df["region"].str.strip().str.title()
df["budget_million"] = df["budget_million"].fillna(df["budget_million"].mean())
df = df.drop_duplicates()

### Exercise 1 — Average by region  ·  *easy*

Average `employment_rate` per region.

In [12]:
print(df.groupby("region")["employment_rate"].mean())

region
Central    92.80
East       93.00
North      94.64
South      91.58
West       88.46
Name: employment_rate, dtype: float64


### Exercise 2 — Sort the result  ·  *easy*

Same as above, but sorted highest first.

In [13]:
print(df.groupby("region")["employment_rate"].mean().sort_values(ascending=False))

region
North      94.64
East       93.00
Central    92.80
South      91.58
West       88.46
Name: employment_rate, dtype: float64


### Exercise 3 — Two aggregations at once  ·  *medium*

Per region, compute average employment and total population using `.agg`.

In [14]:
summary = df.groupby("region").agg(
    avg_employment=("employment_rate", "mean"),
    total_population=("population", "sum"),
).reset_index()
summary

,region,avg_employment,total_population
0,Central,92.80,612500
1,East,93.00,500250
2,North,94.64,510125
3,South,91.58,647000
4,West,88.46,552500


### Exercise 4 — Count per group  ·  *medium*

How many rows (years) does each region have?

In [15]:
print(df.groupby("region")["year"].count())   # 10 each

region
Central    10
East       10
North      10
South      10
West       10
Name: year, dtype: int64


### Exercise 5 — Load and merge the lookup  ·  *medium*

Read the region lookup and left-merge it onto `df` on `region`.

In [16]:
lookup = pd.read_csv("session6_region_lookup.csv")
merged = pd.merge(df, lookup, on="region", how="left")
merged[["region","year","employment_rate","zone","target_employment"]].head()

,region,year,employment_rate,zone,target_employment
0,North,2021,92.3,Zone-A,93.0
1,South,2021,88.7,Zone-B,92.0
2,East,2021,91.2,Zone-A,93.5
3,West,2021,85.4,Zone-C,90.0
4,Central,2021,90.1,Zone-B,93.0


### Exercise 6 — Above target?  ·  *medium*

On the merged table, make a boolean column `above_target` (employment_rate ≥ target_employment) and count how many rows are above.

In [17]:
merged["above_target"] = merged["employment_rate"] >= merged["target_employment"]
print(merged["above_target"].sum())

24


### Exercise 7 — Biggest total budget  ·  *hard*

Which region has the highest **total** budget across all years? (Hint: groupby sum, then `.idxmax()`.)

In [18]:
totals = df.groupby("region")["budget_million"].sum()
print(totals.idxmax())   # the region name

South


### Exercise 8 — Group by a merged column  ·  *hard*

After merging, find the average employment_rate per **zone**.

In [19]:
by_zone = merged.groupby("zone")["employment_rate"].mean()
print(by_zone)

zone
Zone-A    93.82
Zone-B    92.19
Zone-C    88.46
Name: employment_rate, dtype: float64


### Exercise 9 — The full report  ·  *hard*

Build a per-region report: average employment, the target, and a `meets_target` yes/no. Fill in the three blanks.

In [20]:
report = merged.groupby("region").agg(
    avg_employment=("employment_rate", "mean"),
    target=("target_employment", "first"),
).reset_index()
report["meets_target"] = report["avg_employment"] >= report["target"]
report

,region,avg_employment,target,meets_target
0,Central,92.80,93.0,False
1,East,93.00,93.5,False
2,North,94.64,93.0,True
3,South,91.58,92.0,False
4,West,88.46,90.0,False


---

**Done.** Commit this notebook to your GitHub Session-6 folder with a short note.